In [40]:
import yaml
import os
import cv2
import pprint

# 获取所有图片的绝对路径

files = ['./images/' + i for i in os.listdir('./images')]
pprint.pprint(files)

['./images/DM_20230728100716_003_jpg.rf.40671913a4a971ad66e7ecb3418b207e_v4.jpg',
 './images/DM_20230728100716_003_jpg.rf.40671913a4a971ad66e7ecb3418b207e_v4.json',
 './images/DM_20230728100716_006_jpg.rf.834d58841c88524529279b3699b72484_v4.jpg',
 './images/DM_20230728100716_006_jpg.rf.834d58841c88524529279b3699b72484_v4.json',
 './images/DM_20230728101036_002_jpg.rf.3a6f316cb8f00fc60765c5b0558218cf_v4.jpg',
 './images/DM_20230728101036_002_jpg.rf.f636a9cbecea7e1a271d34c457474199_v4.jpg',
 './images/DM_20230728101036_002_jpg.rf.f636a9cbecea7e1a271d34c457474199_v4.json',
 './images/DM_20230728101903_002_jpg.rf.38fefb5b9191485dba20b7d6b3145d8e_v4.jpg',
 './images/DM_20230728101903_002_jpg.rf.38fefb5b9191485dba20b7d6b3145d8e_v4.json',
 './images/DM_20230728101903_002_jpg.rf.644f8451f49c00ce2562b92d8e4dd45a_v4.jpg',
 './images/DM_20230728101903_002_jpg.rf.bcfa8fce696a3c3d123c1d08bca90675_v4.jpg',
 './images/DM_20230728101903_003_jpg.rf.f4c5044c3eb1688570e0fc71611e1b88_v4.jpg',
 './images/D

In [41]:
# 绝对路径化, 并剔除无标签图片

images = []
labels = []
for file in files:
    abs_file_name = os.path.abspath(file)
    if '.json' in abs_file_name:
        labels.append(abs_file_name)
        images.append(abs_file_name[0: -4] + 'jpg')
        
pprint.pprint([[os.path.isfile(n), n] for n in images])

[[True,
  'D:\\Desktop\\TqZzMiao-AwA\\AwA\\yaml\\images\\DM_20230728100716_003_jpg.rf.40671913a4a971ad66e7ecb3418b207e_v4.jpg'],
 [True,
  'D:\\Desktop\\TqZzMiao-AwA\\AwA\\yaml\\images\\DM_20230728100716_006_jpg.rf.834d58841c88524529279b3699b72484_v4.jpg'],
 [True,
  'D:\\Desktop\\TqZzMiao-AwA\\AwA\\yaml\\images\\DM_20230728101036_002_jpg.rf.f636a9cbecea7e1a271d34c457474199_v4.jpg'],
 [True,
  'D:\\Desktop\\TqZzMiao-AwA\\AwA\\yaml\\images\\DM_20230728101903_002_jpg.rf.38fefb5b9191485dba20b7d6b3145d8e_v4.jpg'],
 [True,
  'D:\\Desktop\\TqZzMiao-AwA\\AwA\\yaml\\images\\DM_20230728101903_003_jpg.rf.f4c5044c3eb1688570e0fc71611e1b88_v4.jpg'],
 [True,
  'D:\\Desktop\\TqZzMiao-AwA\\AwA\\yaml\\images\\DM_20230728101903_004_jpg.rf.718f0decc5b1680d4dab2befb2e61e1d_v4.jpg']]


In [42]:
"""进行训练数据集划分"""


import os

# 构建目录
dirs = ['train_data', 
        'train_data/train', 
        'train_data/val', 
        'train_data/train/images', 
        'train_data/train/labels', 
        'train_data/val/images', 
        'train_data/val/labels'
]
for name in dirs:
    try:
        os.mkdir(name)
    except FileExistsError:
        pass

In [43]:
# 划分数据集
import random

train = []  # 训练集
val = []    # 验证集

num       = len(images)  # 总数量
train_num = int(num * 0.6)  # 训练集数量
val_num   = int(num * 0.4)  # 验证集数量

train_num += num - (train_num + val_num)  # 将余数加入训练集中


for _ in range(train_num):
    data = [
        images.pop(random.randint(0, len(images) - 1)),
        labels.pop(random.randint(0, len(labels) - 1))
    ]
    train.append(data)

for _ in range(val_num):
    data = [
        images.pop(random.randint(0, len(images) - 1)),
        labels.pop(random.randint(0, len(labels) - 1))
    ]
    val.append(data)

print(len(train), len(val))

4 2


In [61]:
# 保存数据集


import json

class AwA:
    def __init__(self):
        self.contents = []
        
    def add(self, path, target_path):
        content = json.load(open(path, 'r', encoding='utf-8'))  # 打开图片
        imageSize = [content['imageWidth'], content['imageHeight']]    # 图片大小
        shapes = {'target_path': target_path + os.path.basename(path), 'content': []}
        for shape in content['shapes']:
            shapes['content'].append(
                [
                    shape['label'],                        # 标签
                    shape['points'][0][0] / imageSize[0],  # 左上角x坐标
                    shape['points'][0][1] / imageSize[1],  # 左上角y坐标
                    shape['points'][1][0] / imageSize[0],  # 右下角x坐标
                    shape['points'][0][1] / imageSize[1]   # 右下角y坐标
                ]
            )
        self.contents.append(shapes)

    def save(self):
        """保存所有标签并构建yaml"""
        # 转文本
        tabels = []  # 标签列表
        for shapes in self.contents:
            text = ''
            for shape in shapes['content']:
                if shape[0] not in tabels:
                    tabels.append(shape[0])
                    
                text += f'{tabels.index(shape[0])} {shape[1]} {shape[2]} {shape[3]} {shape[4]}\n'
            
            open(shapes['target_path'], 'w', encoding='utf-8').write(text)

        yaml_data = {
            'train': 'train_data/train/images',
            'val':   'train_data/val/images',
            'names':  tabels,
            'nc':     len(tabels)
        }

        f = open('./train_config.yaml', 'w', encoding='utf-8')
        yaml.dump(yaml_data, f, allow_unicode=True)
        

awa = AwA()

# 构建训练集
for file in train:
    file_content = open(file[0], 'rb').read()
    open(
        './train_data/train/images/' + os.path.basename(file[0]),
        'wb'
    ).write(file_content)

    awa.add(file[1], './train_data/train/labels')

# 构建验证集
for file in val:
    file_content = open(file[0], 'rb').read()
    open(
        './train_data/val/images/' + os.path.basename(file[0]),
        'wb'
    ).write(file_content)

    awa.add(file[1], './train_data/val/labels')
awa.save()


In [62]:
dir(os)

['DirEntry',
 'F_OK',
 'GenericAlias',
 'Mapping',
 'MutableMapping',
 'O_APPEND',
 'O_BINARY',
 'O_CREAT',
 'O_EXCL',
 'O_NOINHERIT',
 'O_RANDOM',
 'O_RDONLY',
 'O_RDWR',
 'O_SEQUENTIAL',
 'O_SHORT_LIVED',
 'O_TEMPORARY',
 'O_TEXT',
 'O_TRUNC',
 'O_WRONLY',
 'P_DETACH',
 'P_NOWAIT',
 'P_NOWAITO',
 'P_OVERLAY',
 'P_WAIT',
 'PathLike',
 'R_OK',
 'SEEK_CUR',
 'SEEK_END',
 'SEEK_SET',
 'TMP_MAX',
 'W_OK',
 'X_OK',
 '_AddedDllDirectory',
 '_Environ',
 '__all__',
 '__builtins__',
 '__cached__',
 '__doc__',
 '__file__',
 '__loader__',
 '__name__',
 '__package__',
 '__spec__',
 '_check_methods',
 '_execvpe',
 '_exists',
 '_exit',
 '_fspath',
 '_get_exports_list',
 '_walk',
 '_wrap_close',
 'abc',
 'abort',
 'access',
 'add_dll_directory',
 'altsep',
 'chdir',
 'chmod',
 'close',
 'closerange',
 'cpu_count',
 'curdir',
 'defpath',
 'device_encoding',
 'devnull',
 'dup',
 'dup2',
 'environ',
 'error',
 'execl',
 'execle',
 'execlp',
 'execlpe',
 'execv',
 'execve',
 'execvp',
 'execvpe',
 'exts

Help on function dump in module yaml:

dump(data, stream=None, Dumper=<class 'yaml.dumper.Dumper'>, **kwds)
    Serialize a Python object into a YAML stream.
    If stream is None, return the produced string instead.

